In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

dbutils.widgets.removeAll()

## PARAMETRIZAR CATALOGO A PROD
dbutils.widgets.text("PRM_catalogo","catalogo_desa_intEcommerce")

PRM_catalogo = dbutils.widgets.get("PRM_catalogo")

def read_tablaecommerce():
    df_interaccion = spark.table(f"{PRM_catalogo}.bronze.ecommerce_data") \
        .select(
            trim(col("Interaction_id_system")).alias("ID_interaccion"), 
            trim(col("Interaction_date")).alias("Fecha_Interaccion"), 
            trim(col("User_id_system")).alias("ID_Cliente"),  
            trim(col("Product_id")).alias("Cod_producto"),
            trim(col("TypeInt_id")).alias("Cod_Tipo_Interaccion"),
            trim(col("Event")).alias("Evento"),      
            trim(col("Location")).alias("Locacion"), 
           ##
            when (col("Quantity") == '-', 'NO APLICA')
            .otherwise(trim(col("Quantity"))).alias("Cantidad_Producto"),
           ##
            col("Product_rating").alias("Puntuacion_Producto"),                
            to_timestamp(date_format(current_timestamp(), "yyyy-MM-dd")).alias("Fecha_proceso")             
        )    
    return df_interaccion



def CleanHist_IntEcommerce(df_interaccion):
    return df_interaccion.where(
        col("ID_interaccion").isNotNull() & col("Fecha_Interaccion").isNotNull() & col("Locacion").isNotNull()
    ).dropDuplicates(["ID_interaccion", "Fecha_Interaccion", "Locacion"])



def main():
 
    df_interaccion = read_tablaecommerce()
 
    df_interaccion_final = CleanHist_IntEcommerce(df_interaccion)

    df_interaccion_final.write \
    .mode("overwrite") \
    .saveAsTable(f"{PRM_catalogo}.silver.Tabla_IntEcommerce")


main()
